# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 8: Advanced LLM Agents</font>

# <font color="#003660">LLM Agent Plannung - ToDo Lists</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how LLM agents are build with LangChain and LangGraph. <br>
        ... will know how states in LLM agents work. <br>
        ... will know how states can be used for ToDo Lists as planning and progress indicator in LLM agents. <br>
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Langchain-AI Deep Agents from Scratch](https://github.com/langchain-ai/deep-agents-from-scratch)
* [LangChain Academy](https://academy.langchain.com/)
* [Introduction to LangChain Agents](https://github.com/langchain-ai/langchain-academy/blob/main/module-1/agent.ipynb)
* [LangChain Docs (Python)](https://python.langchain.com/)

In [ ]:
!pip install -U wikipedia langchain langchain-community langchain-openai

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [ ]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull gpt-oss:20b # takes around 2 minutes

## Planning: TODO Lists

![Screenshot 2025-08-15 at 11.41.46 AM.png](https://github.com/langchain-ai/deep-agents-from-scratch/blob/main/notebooks/assets/agent_header_todo.png?raw=true)

Many agents use TODO lists as a critical navigation tool for steering through long-running, complex tasks. Claude Code leverages [plan mode](https://www.anthropic.com/engineering/claude-code-best-practices) to create structured TODO lists before executing tasks, utilizing a specific tool called `TodoWrite` based on the [Claude Code prompt](https://cchistory.mariozechner.at/). Each TODO item contains two key components: content (a short, specific task description) and status (pending, in_progress, or completed).

The challenge with TODO lists lies in maintaining attention as context windows grow—the average Manus task uses approximately 50 tool calls, creating substantial risk of [context rot](https://research.trychroma.com/context-rot). Agents become vulnerable to drifting off-topic or forgetting earlier objectives during lengthy conversations or complicated tasks. By continuously rewriting and updating the TODO list, agents like Manus effectively recite their objectives at the end of the context, helping to [stay focused on task](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus) and prevent mission drift.

<!-- The style below reduces the gap between items in the same bulleted list. Run once per notebook -->
<style>
/* JupyterLab + classic notebook */
.jp-RenderedHTMLCommon ul, .text_cell_render ul { margin-top: .25em; margin-bottom: .35em; padding-left: 1.2em; }
.jp-RenderedHTMLCommon ul ul, .text_cell_render ul ul { margin-top: .15em; margin-bottom: .15em; padding-left: 1.0em; }
.jp-RenderedHTMLCommon li, .text_cell_render li { margin: .1em 0; }
</style>

### State

Just as in the previous notebook, you will be using a `create_agent` with custom state.

The state object serves as our primary mechanism for storing and passing context between different phases of the workflow. The State consists of the schema of the graph as well as reducer functions which specify how to apply updates to the state.

There are three primary elements defined in the DeepAgent scheme: **`messages`**, **`todo`** and **`files`**.  

- **`messages`** are inherited from `AgentState`, which was described in the first lesson.
  - The `add_messages` reducer will append new messages to the end of the message list.  
- **`todo`** are a list of `Todo` tasks. Each task has a description: `content` and a `status`: pending, in_progress, completed.
  - No custom reducer is defined, so updates overwrite the list on write.
- **`files`** is a virtual file system contained in state which you will explore in the next lesson.  


In [ ]:
"""State management for deep agents with TODO tracking and virtual file systems.

This module defines the extended agent state structure that supports:
- Task planning and progress tracking through TODO lists
- Context offloading through a virtual file system stored in state
- Efficient state merging with reducer functions
"""

from typing import Annotated, Literal, NotRequired
from typing_extensions import TypedDict

from langchain.agents import AgentState

class Todo(TypedDict):
    """A structured task item for tracking progress through complex workflows.

    Attributes:
        content: Short, specific description of the task
        status: Current state - pending, in_progress, or completed
    """

    content: str
    status: Literal["pending", "in_progress", "completed"]


def file_reducer(left, right):
    """Merge two file dictionaries, with right side taking precedence.

    Used as a reducer function for the files field in agent state,
    allowing incremental updates to the virtual file system.

    Args:
        left: Left side dictionary (existing files)
        right: Right side dictionary (new/updated files)

    Returns:
        Merged dictionary with right values overriding left values
    """
    if left is None:
        return right
    elif right is None:
        return left
    else:
        return {**left, **right}


class DeepAgentState(AgentState):
    """Extended agent state that includes task tracking and virtual file system.

    Inherits from LangGraph's AgentState and adds:
    - todos: List of Todo items for task planning and progress tracking
    - files: Virtual file system stored as dict mapping filenames to content
    """

    todos: NotRequired[list[Todo]]
    files: Annotated[NotRequired[dict[str, str]], file_reducer]

### Tool Description  - Todo Tool
As described above, long-running agents can use a todo list to stay on task. To enable that, todo tools, `write_todo` and `read_todo` are created. The tool description below is provided to the LLM detailing when to use the todo list, what it contains, and how to read and update it.   
Note, that while the list contains individual tasks, it is updated with a full rewrite of the list. This allows the LLM to reconsider tasks as it makes progress.



In [ ]:
WRITE_TODOS_DESCRIPTION = """Create and manage structured task lists for tracking progress through complex workflows.

## When to Use
- Multi-step or non-trivial tasks requiring coordination
- When user provides multiple tasks or explicitly requests todo list
- Avoid for single, trivial actions unless directed otherwise

## Structure
- Maintain one list containing multiple todo objects (content, status, id)
- Use clear, actionable content descriptions
- Status must be: pending, in_progress, or completed

## Best Practices
- Only one in_progress task at a time
- Mark completed immediately when task is fully done
- Always send the full updated list when making changes
- Prune irrelevant items to keep list focused

## Progress Updates
- Call TodoWrite again to change task status or edit content
- Reflect real-time progress; don't batch completions
- If blocked, keep in_progress and add new task describing blocker

## Parameters
- todos: List of TODO items with content and status fields

## Returns
Updates agent state with new todo list."""

### Write and Read ToDo Tools

Below, you will create the **`write_todos`** and **`read_todos`** tools:  
The **`write_todos`** tool takes a todo list from the LLM as an argument and writes it to state, overwriting any previous list. It then returns a `ToolMessage` containing the list that was written. Note that writing the list makes the information available to the LLM in the conversation history stored in `messages`, both in the LLM-generated tool call and in the returned `ToolMessage`.  

The **`read_todos`** tool reads the todo list from state and returns it as a `ToolMessage`. It can be used to refresh the information in the LLM context.  


Note that these tools will use some of the features that were discussed in the previous lesson:

- `InjectedState` to provide the tool access to the graph state.
- `Command` to update values in state.


In [ ]:
"""TODO management tools for task planning and progress tracking.

This module provides tools for creating and managing structured task lists
that enable agents to plan complex workflows and track progress through
multi-step operations.
"""

from typing import Annotated

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command

@tool(description=WRITE_TODOS_DESCRIPTION,parse_docstring=True)
def write_todos(
    todos: list[Todo], tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """Create or update the agent's TODO list for task planning and tracking.

    Args:
        todos: List of Todo items with content and status
        tool_call_id: Tool call identifier for message response

    Returns:
        Command to update agent state with new TODO list
    """
    return Command(
        update={
            "todos": todos,
            "messages": [
                ToolMessage(f"Updated todo list to {todos}", tool_call_id=tool_call_id)
            ],
        }
    )


@tool(parse_docstring=True)
def read_todos(
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """Read the current TODO list from the agent state.

    This tool allows the agent to retrieve and review the current TODO list
    to stay focused on remaining tasks and track progress through complex workflows.

    Args:
        state: Injected agent state containing the current TODO list
        tool_call_id: Injected tool call identifier for message tracking

    Returns:
        Formatted string representation of the current TODO list
    """
    todos = state.get("todos", [])
    if not todos:
        return "No todos currently in the list."

    result = "Current TODO List:\n"
    for i, todo in enumerate(todos, 1):
        status_emoji = {"pending": "⏳", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result += f"{i}. {emoji} {todo['content']} ({todo['status']})\n"

    return result.strip()

IMPORTANT: those are not stored anywhere but in the state. No file storage on your system. Not persistent.

### Graph  

As in the first lesson, you'll build an agent using `create_agent`.  We're going to focus on getting our agent to use a todo list. We'll follow the Manus approach of having a todo recitation after each task, and we'll use the tools you just created above.

In [ ]:
TODO_USAGE_INSTRUCTIONS = """Based upon the user's request:
1. Use the write_todos tool to create TODO at the start of a user request, per the tool description.
2. After you accomplish a TODO, use the read_todos to read the TODOs in order to remind yourself of the plan.
3. Reflect on what you've done and the TODO.
4. Mark you task as completed, and proceed to the next TODO.
5. Continue this process until you have completed all TODOs.

IMPORTANT: Always create a research plan of TODOs and conduct research following the above guidelines for ANY user request.
IMPORTANT: Aim to batch research tasks into a *single TODO* in order to minimize the number of TODOs you have to keep track of.
"""

In [ ]:
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.retrievers import WikipediaRetriever

# Mock search tool
@tool(parse_docstring=True)
def search_in_wikipedia(query: str) -> str:
    """Search Wikipedia for a given query.

    This tool searches queries like 'Jon Snow' in Wikipedia and returns important passages from Wikipedia articles.

    Args:
        query: The search query string. Be specific and clear about what information you're looking for.

    Returns:
        Search results from wikipedia."""
    retriever = WikipediaRetriever()
    docs = retriever.invoke(query)
    results = "\n\n-----\n\n".join([f"Document {i}:\n\nMetadata:\n-Title: {docs[i].metadata['title'].strip()}\n-Path: https://en.wikipedia.org/wiki/{docs[i].metadata['title'].strip().replace(' ', '_')}\n\nContent:\n{docs[i].metadata['summary'].strip()}" for i in range(len(docs))])

    return results


# Create agent using create_react_agent directly
config = {
    "model": "gpt-oss",
    "base_url": "http://127.0.0.1:11434/v1",
	"api_key": "ollama",
    "max_tokens": 8192,
    "temperature": 0,
    "seed": 42,
}

model = ChatOpenAI(
    **config
)
tools = [write_todos, search_in_wikipedia, read_todos]

# Create agent
agent = create_agent(
    model,
    tools,
    system_prompt=TODO_USAGE_INSTRUCTIONS,
    state_schema=DeepAgentState,
)

# Show the agent
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

Start the graph with no `todos` in state and an user research request.

In [ ]:
# Example usage
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "First search on Wikipedia for brown bears. Then search on wikipedia for polar bears. Finally, explain the differences.",
            }
        ],
        "todos": [],
    }
)

for m in result["messages"]:
    m.pretty_print()

## Filesystem, Subagents and other Extensions

We will now switch to a simpler version of implementing deep agents based on what is called `middleware`.

If you are interested to implement the complete deep agent from scratch you can follow this tutorial which heavily inspired this notbook: [Deep Agents from Scratch](https://github.com/langchain-ai/deep-agents-from-scratch/)